In [1]:
import import_ipynb
import numpy as np
import sys
from itertools import combinations, combinations_with_replacement

from functions import *

In [2]:
"""Create a grid of momenta"""

momentum_grid = [[px, py, pz] for pz in range(-5, 6) for py in range(-5, 6) for px in range(-5, 6)\
                 if (pz==0) and (px>0) and ((np.sqrt(px**2+py**2+pz**2)<=5))]
momentum_grid = np.array(momentum_grid)

In [3]:
"""Convert the momenta and kinetic energy grids to dictionaries"""

momentum_dict = {i: item for i, item in enumerate(momentum_grid)}
kinetic_energy_dict = {key: np.sqrt(val@val) for key, val in momentum_dict.items()}

In [4]:
"""Create array of allowed momenta for 2->2 scattering"""

# create list of momentum pairs where p1 not distinguishable from p2 and p3 not distinguishable from p4
# remove any p1=p3
all_mom_combinations=list(combinations_with_replacement(list(combinations_with_replacement([*range(35)], 2)), 2))
all_mom_combinations = [[item[0][0], item[0][1], item[1][0], item[1][1]] for item in all_mom_combinations if item[0][0] != item[1][0]]

# select for pairs that conserve momentum and kinetic energy and g-function is nonzero
ok_mom_combinations=[item for item in all_mom_combinations\
 if (np.sum(np.abs(momentum_dict[item[0]]+momentum_dict[item[1]]-momentum_dict[item[2]]-momentum_dict[item[3]])) < 10**-7)\
 and (np.abs(kinetic_energy_dict[item[0]]+kinetic_energy_dict[item[1]]-kinetic_energy_dict[item[2]]-kinetic_energy_dict[item[3]]) < 10**-7)\
 and (np.abs(g_factor(momentum_dict[item[0]], momentum_dict[item[1]], momentum_dict[item[2]], momentum_dict[item[3]])) > 10**-7)]

# generate reverse pairings, e.g., (q1, q2, q3, q4) -> (q1, q2, q3, q4), (q3, q4, q1, q2)
mom_combinations = []
for item in ok_mom_combinations: 
    mom_combinations.append(item)
    mom_combinations.append([*item[2:], *item[:2]])
mom_combinations.sort()

mom_combinations=np.array(mom_combinations)

In [5]:
mom_combinations

array([[ 0, 33,  1, 32],
       [ 0, 34,  2, 32],
       [ 1, 13,  3, 10],
       [ 1, 32,  0, 33],
       [ 1, 34,  2, 33],
       [ 2,  9,  4,  6],
       [ 2, 12,  6,  7],
       [ 2, 22,  3, 19],
       [ 2, 32,  0, 34],
       [ 2, 33,  1, 34],
       [ 3, 10,  1, 13],
       [ 3, 19,  2, 22],
       [ 3, 29,  4, 28],
       [ 3, 30,  5, 28],
       [ 3, 31,  6, 28],
       [ 4,  6,  2,  9],
       [ 4, 12,  7,  9],
       [ 4, 28,  3, 29],
       [ 4, 30,  5, 29],
       [ 4, 31,  6, 29],
       [ 5, 28,  3, 30],
       [ 5, 29,  4, 30],
       [ 5, 31,  6, 30],
       [ 6,  7,  2, 12],
       [ 6, 21,  7, 19],
       [ 6, 28,  3, 31],
       [ 6, 29,  4, 31],
       [ 6, 30,  5, 31],
       [ 7,  9,  4, 12],
       [ 7, 19,  6, 21],
       [ 7, 25,  8, 24],
       [ 7, 26,  9, 24],
       [ 7, 27, 10, 24],
       [ 8, 24,  7, 25],
       [ 8, 26,  9, 25],
       [ 8, 27, 10, 25],
       [ 9, 24,  7, 26],
       [ 9, 25,  8, 26],
       [ 9, 27, 10, 26],
       [10, 24,  7, 27],


In [28]:
kec_nt = np.array(kec_nt)
Nkec = len(kec_nt)

#ninit = np.sort(np.array([0,5,10,25,28,33])) # N=6
ninit = np.sort(np.array([0,5,10,12,20,25,28,33])) # N=8
#ninit = np.sort(np.array([0,5,8,10,12,20,25,26,28,33])) # N=10

Nn = len(ninit)

pinit = np.sum(ps[ninit], axis=0)
keinit = np.sum(kes[ninit])

In [29]:
print(f'Reference state is {ninit}')
print(f'Reference state total momentum: {pinit}')
print(f'Reference state kinetic energy: {keinit}')

Reference state is [ 0  5 10 12 20 25 28 33]
Reference state total momentum: [16.  0.  0.]
Reference state kinetic energy: 26.95100454752356


In [30]:
binom = np.array(list(combinations([i for i in range(Nn)],2)))
Nbinom = len(binom)

In [31]:
def check(state):
    ans = True
    for i in range(len(state)-2):
        if np.var(state[i:i+3]) < 1e-7:
            ans = False
    return ans

In [32]:
def apply(state):
    newstate = []
    for i in range(Nbinom):
        k = np.array([state[binom[i,0]], state[binom[i,1]]])
        for j in range(Nkec):
            if np.sum(np.abs(k-kec_nt[j,:2]))==0:
                state_i = state.copy()
                state_i[binom[i,0]] = kec_nt[j,2]
                state_i[binom[i,1]] = kec_nt[j,3]
                state_i = np.sort(state_i)
                if check(state_i):
                    newstate.append(state_i)
    return np.array(newstate)

In [33]:
nnewstate = 10  # > 1 to enter the while loop
p_states = np.array([ninit])
newstate1 = np.array([np.zeros(Nn), ninit])
trial = 0

while nnewstate > 1:
    print(f'---------------- H^{trial+1} ----------------')
    newstate2 = np.zeros((1,Nn))
    #newstate3 = np.zeros((1,Nn))
    for i in range(1,len(newstate1)):
        newi = apply(newstate1[i])
        if len(newi) > 0:
            newstate2 = np.append(newstate2, newi, axis=0)
       
    newstate1 = np.array([np.zeros(Nn)])
    for j in range(1,len(newstate2)):
        dist = np.sum(np.abs(p_states - newstate2[j]), axis=1)
        if np.min(dist) > 1e-7:
            p_states = np.append(p_states, [newstate2[j].astype(int)], axis=0)
            newstate1 = np.append(newstate1, [newstate2[j]], axis=0)
    nnewstate = len(newstate1)
    print(f'Number of new states(mod flavor choice) at this round: {nnewstate-1}')
    print(f'Number of states(mod flavor choice) visited so far: {len(p_states)}')
        
    trial += 1

---------------- H^1 ----------------
Number of new states(mod flavor choice) at this round: 4
Number of states(mod flavor choice) visited so far: 5
---------------- H^2 ----------------
Number of new states(mod flavor choice) at this round: 8
Number of states(mod flavor choice) visited so far: 13
---------------- H^3 ----------------
Number of new states(mod flavor choice) at this round: 10
Number of states(mod flavor choice) visited so far: 23
---------------- H^4 ----------------
Number of new states(mod flavor choice) at this round: 6
Number of states(mod flavor choice) visited so far: 29
---------------- H^5 ----------------
Number of new states(mod flavor choice) at this round: 1
Number of states(mod flavor choice) visited so far: 30
---------------- H^6 ----------------
Number of new states(mod flavor choice) at this round: 0
Number of states(mod flavor choice) visited so far: 30


In [34]:
n_states = 0

for i in range(len(p_states)):
    totp = np.sum(ps[p_states[i]], axis=0)
    totke = np.sum(kes[p_states[i]])
    nstate = 2**Nn/4**(Nn-len(set(p_states[i])))
    strs = ' '.join([str(x) for x in p_states[i]]) + '\n'
    n_states += nstate
    if np.sum(np.abs(totp-pinit)) > 1e-7 or np.abs(totke-keinit) > 1e-7:
        print('Total momentum or kinetic energy is not conserved!')

In [35]:
print(f'Number of momentum modes pair with conserved P and E is {len(p_states)}')
print(f'Number of states with conserved P,E, and arbitrary flavor contents is {n_states:.0f}')
bins_visited = np.sort(np.array(list(set(p_states.flatten()))))
print(f'Activated momentum modes:')
print(bins_visited)

Number of momentum modes pair with conserved P and E is 30
Number of states with conserved P,E, and arbitrary flavor contents is 6528
Activated momentum modes:
[ 0  1  3  5  8 10 11 12 13 20 21 22 25 27 28 30 32 33]
